# 03 — SHAP Global cho XGBoost

**Mục đích:** Tài liệu này trình bày kết quả giải thích toàn cục (global explanation) bằng SHAP cho mô hình XGBoost tham số mặc định, huấn luyện trên bộ đặc trưng feature freeze v1 với dữ liệu thật. Kết quả gồm Summary Plot (beeswarm) và Bar Plot mean(|SHAP|), là cơ sở cho các bước tiếp theo: Dependence Plot, định nghĩa nhóm đặc trưng cho reason codes (Tuần 2 – T4) và giải thích cục bộ (Tuần 2 – T5).

**Tài liệu tham chiếu:** `reports/project_plan.md` (mục 3.5e và Tuần 2 – T3), `configs/feature_freeze_v1.yaml`, `notebooks/03_shap_prototype.ipynb`.

**Thiết lập:**

| Hạng mục | Giá trị |
|---|---|
| Mô hình | `xgboost_default` — `src.train.get_advanced_default_models`, pipeline `src.pipelines.make_pipeline` (không có SEX) |
| Dữ liệu huấn luyện | Tập Train (60%, `data/splits/splits.json`) |
| Mẫu giải thích | 3.000 dòng lấy ngẫu nhiên từ tập Valid (`random_state = 42`) |
| Explainer | `shap.TreeExplainer`, `feature_perturbation = "tree_path_dependent"` |
| Thang giá trị SHAP | Log-odds (margin) của mô hình chưa hiệu chuẩn |

Tập Valid được chọn làm mẫu giải thích để mô tả hành vi của mô hình trên dữ liệu không tham gia huấn luyện; SHAP không sử dụng nhãn nên việc này không ảnh hưởng đến vai trò của tập Valid trong calibration và chọn ngưỡng. Tập Test không được sử dụng (theo quy định mục 8.2).

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
import sklearn
import xgboost as xgb
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.config import load_config  # noqa: E402
from src.data_split import load_split_data  # noqa: E402
from src.features import build_features  # noqa: E402
from src.pipelines import make_pipeline  # noqa: E402
from src.train import get_advanced_default_models  # noqa: E402

SEED = load_config()["random_state"]
N_SAMPLE = 3000  # trong khoảng 2.000–5.000 dòng theo project_plan.md
MAX_DISPLAY = 20
FIG_DIR = PROJECT_ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.float_format", "{:.4f}".format)
print("shap", shap.__version__, "| xgboost", xgb.__version__, "| sklearn", sklearn.__version__)

## 1. Dữ liệu và mô hình

Mô hình được huấn luyện lại trên toàn bộ tập Train với cùng factory và pipeline mà `scripts/run_pipeline.py --mode advanced` sử dụng để đăng ký `xgboost_default` trên MLflow. Do `random_state` được cố định, mô hình thu được tương đương bản đã log, đồng thời notebook có thể chạy lại từ bản clone sạch mà không phụ thuộc thư mục `mlruns/` cục bộ.

In [ ]:
train_df, valid_df, _ = load_split_data()
X_train, y_train = build_features(train_df)
X_valid, y_valid = build_features(valid_df)

pipe = make_pipeline(clone(get_advanced_default_models(SEED)["xgboost_default"]))
pipe.fit(X_train, y_train)

prep, model = pipe[:-1], pipe[-1]
col_tf = pipe.named_steps["preprocess"].named_steps["preprocessor"]

print(f"Train: {X_train.shape} | Valid: {X_valid.shape}")
print(f"ROC-AUC trên Valid (tham chiếu): {roc_auc_score(y_valid, pipe.predict_proba(X_valid)[:, 1]):.4f}")

## 2. Mẫu giải thích

Mẫu 3.000 dòng được lấy ngẫu nhiên từ tập Valid. Dữ liệu đưa vào explainer là đầu ra của bước tiền xử lý (one-hot cho biến phân loại, impute median và chuẩn hóa cho biến số). Để biểu đồ hiển thị giá trị gốc, các cột số được đưa về thang ban đầu bằng `inverse_transform` của `StandardScaler`; giá trị SHAP không thay đổi vì chỉ phụ thuộc vào dữ liệu đầu vào của mô hình.

In [ ]:
X_sample = X_valid.sample(N_SAMPLE, random_state=SEED)
y_sample = y_valid.loc[X_sample.index]

X_model = prep.transform(X_sample)
feature_names = [name.split("__", 1)[1] for name in col_tf.get_feature_names_out()]

# Cột số: đưa về thang gốc để hiển thị (one-hot giữ nguyên 0/1).
cat_cols = list(col_tf.transformers_[0][2])
num_cols = list(col_tf.transformers_[1][2])
n_cat_out = len(feature_names) - len(num_cols)
X_display = pd.DataFrame(X_model.copy(), columns=feature_names, index=X_sample.index)
X_display[num_cols] = col_tf.named_transformers_["numeric"]["scaler"].inverse_transform(X_model[:, n_cat_out:])

print(f"Mẫu: {len(X_sample)} dòng | tỷ lệ vỡ nợ: {y_sample.mean():.3f} (Valid: {y_valid.mean():.3f})")
print(f"Số cột sau tiền xử lý: {len(feature_names)} ({n_cat_out} cột one-hot từ {len(cat_cols)} biến phân loại, {len(num_cols)} biến số)")

## 3. Tính giá trị SHAP

`TreeExplainer` tính SHAP chính xác cho mô hình cây. Tính cộng tính được kiểm tra trên thang margin: $\phi_0 + \sum_i \phi_i = m(x)$, với $m(x)$ là đầu ra `output_margin=True` của XGBoost.

In [ ]:
explainer = shap.TreeExplainer(model)

t0 = time.perf_counter()
sv_raw = explainer(X_model)
elapsed = time.perf_counter() - t0

sv = shap.Explanation(
    values=sv_raw.values,
    base_values=sv_raw.base_values,
    data=X_display.to_numpy(),
    feature_names=feature_names,
)

margin = model.predict(X_model, output_margin=True)
max_err = np.abs(sv.base_values + sv.values.sum(axis=1) - margin).max()
base = float(np.ravel(sv.base_values)[0])

print(f"Shape SHAP: {sv.values.shape} | thời gian: {elapsed:.2f} s")
print(f"Base value: {base:.4f} (margin) ≈ PD {1 / (1 + np.exp(-base)):.3f}")
print(f"Sai số cộng tính lớn nhất: {max_err:.2e}")
assert max_err < 1e-3

## 4. Summary Plot (beeswarm)

Mỗi điểm là một khách hàng trong mẫu. Trục hoành là giá trị SHAP (thang log-odds): giá trị dương làm tăng xác suất vỡ nợ, giá trị âm làm giảm. Màu thể hiện giá trị của đặc trưng (đỏ: cao, xanh: thấp). Các đặc trưng được sắp xếp theo mean(|SHAP|) giảm dần; biểu đồ hiển thị 20 cột có mức độ quan trọng cao nhất.

In [ ]:
shap.plots.beeswarm(sv, max_display=MAX_DISPLAY, show=False)
plt.title("SHAP Summary Plot — XGBoost (mẫu Valid, n = 3.000)")
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_xgb_summary.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Bar Plot mean(|SHAP|)

### 5.1. Theo cột sau tiền xử lý

Mức độ quan trọng toàn cục của mỗi cột được đo bằng trung bình trị tuyệt đối giá trị SHAP trên mẫu.

In [ ]:
shap.plots.bar(sv, max_display=MAX_DISPLAY, show=False)
plt.title("mean(|SHAP|) theo cột — XGBoost")
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_xgb_bar.png", dpi=150, bbox_inches="tight")
plt.show()

### 5.2. Theo biến gốc

Các biến phân loại (`PAY_1`–`PAY_6`, `EDUCATION`, `MARRIAGE`, `AGE_BIN`) được one-hot thành nhiều cột, khiến mức độ quan trọng của một biến bị chia nhỏ trên Bar Plot ở mục 5.1. Do SHAP có tính cộng tính, giá trị SHAP của các cột one-hot được cộng lại theo từng biến gốc trước khi lấy trị tuyệt đối. Bảng dưới đây là cơ sở chính để xếp hạng mức độ quan trọng của đặc trưng.

In [ ]:
def source_variable(col):
    for var in sorted(cat_cols, key=len, reverse=True):
        if col.startswith(var + "_"):
            return var
    return col


col_to_var = pd.Series({c: source_variable(c) for c in feature_names})
shap_by_var = pd.DataFrame(sv.values, columns=feature_names).T.groupby(col_to_var).sum().T

importance = shap_by_var.abs().mean().sort_values(ascending=False).rename("mean_abs_shap").to_frame()
importance["share"] = importance["mean_abs_shap"] / importance["mean_abs_shap"].sum()
importance["cum_share"] = importance["share"].cumsum()
importance["rank"] = np.arange(1, len(importance) + 1)
importance.to_csv(FIG_DIR / "shap_xgb_importance_by_variable.csv")

sv_var = shap.Explanation(
    values=shap_by_var.to_numpy(),
    base_values=sv.base_values,
    feature_names=list(shap_by_var.columns),
)
shap.plots.bar(sv_var, max_display=MAX_DISPLAY, show=False)
plt.title("mean(|SHAP|) theo biến gốc — XGBoost")
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_xgb_bar_by_variable.png", dpi=150, bbox_inches="tight")
plt.show()

importance.head(MAX_DISPLAY)

Bảng bổ sung: vị trí của các biến nhân khẩu học và tổng đóng góp theo một số họ đặc trưng (tổng share của các biến cùng tiền tố). Cách gom họ ở đây chỉ phục vụ mô tả; định nghĩa nhóm đặc trưng chính thức cho reason codes được chốt tại Tuần 2 – T4.

In [ ]:
families = {
    "Trạng thái trả nợ PAY_1–6": [f"PAY_{i}" for i in range(1, 7)],
    "Tổng hợp trễ hạn (PAY_MAX/MEAN/SLOPE/LATE_*)": ["PAY_MAX", "PAY_MEAN", "PAY_SLOPE", "PAY_LATE_CONSECUTIVE", "PAY_LATE_2PLUS_COUNT"],
    "Số tiền trả PAY_AMT1–6": [f"PAY_AMT{i}" for i in range(1, 7)],
    "Tỷ lệ trả/dư nợ PAY_RATIO_*": [c for c in importance.index if c.startswith("PAY_RATIO_")],
    "Dư nợ BILL_AMT1–6": [f"BILL_AMT{i}" for i in range(1, 7)],
    "Biến động dư nợ (BILL_STD/DELTA)": ["BILL_STD", "BILL_DELTA"],
    "Sử dụng hạn mức UTIL_* + LIMIT_BAL": [c for c in importance.index if c.startswith("UTIL_")] + ["LIMIT_BAL"],
    "Cờ trả tối thiểu MIN_PAY_FLAG_*": [c for c in importance.index if c.startswith("MIN_PAY_FLAG_")],
    "Nhân khẩu học (AGE_BIN/EDUCATION/MARRIAGE)": ["AGE_BIN", "EDUCATION", "MARRIAGE"],
}
assert sorted(sum(families.values(), [])) == sorted(importance.index)

family_share = pd.Series({k: importance.loc[v, "share"].sum() for k, v in families.items()}).sort_values(ascending=False)
display(family_share.rename("share").to_frame())
importance.loc[["AGE_BIN", "EDUCATION", "MARRIAGE"], ["mean_abs_shap", "share", "rank"]]

## 6. Độ ổn định theo mẫu

Thứ hạng mức độ quan trọng (theo biến gốc) được tính lại trên hai mẫu Valid khác (`random_state` = 0 và 1) và một mẫu 3.000 dòng từ tập Train, sau đó so sánh với mẫu chính bằng hệ số tương quan Spearman và mức trùng lặp Top 10.

In [ ]:
def importance_by_variable(X_raw):
    values = explainer.shap_values(prep.transform(X_raw))
    return pd.DataFrame(values, columns=feature_names).T.groupby(col_to_var).sum().T.abs().mean()


ref = importance["mean_abs_shap"]
top10 = set(ref.index[:10])
rows = {}
for label, X_raw in {
    "Valid, seed 0": X_valid.sample(N_SAMPLE, random_state=0),
    "Valid, seed 1": X_valid.sample(N_SAMPLE, random_state=1),
    "Train, seed 42": X_train.sample(N_SAMPLE, random_state=SEED),
}.items():
    other = importance_by_variable(X_raw)[ref.index]
    rows[label] = {
        "Spearman": ref.corr(other, method="spearman"),
        "Trùng Top 10": len(top10 & set(other.sort_values(ascending=False).index[:10])),
    }
pd.DataFrame(rows).T

## 7. Nhận xét

### 7.1. Tính đúng đắn và chi phí tính toán

- Sai số cộng tính lớn nhất trên thang margin ở mức $10^{-6}$; giá trị SHAP khớp với đầu ra của mô hình.
- Base value $\phi_0 = -1{,}403$ (margin), tương ứng PD ≈ 0,197, gần với tỷ lệ vỡ nợ của tập Train (≈ 0,22). Chênh lệch này là bình thường do $\phi_0$ là trung bình trên thang log-odds và mô hình chưa hiệu chuẩn.
- Thời gian tính SHAP cho 3.000 dòng × 119 cột dưới 1 giây, xác nhận `TreeExplainer` đáp ứng yêu cầu cho cả phân tích toàn cục và demo trực tiếp.

### 7.2. Mức độ quan trọng toàn cục

- **Lịch sử trễ hạn chiếm ưu thế.** `PAY_MAX` (mức trễ hạn lớn nhất trong 6 tháng) đứng đầu với mean(|SHAP|) = 0,61, chiếm khoảng 15% tổng mức độ quan trọng và gấp hơn hai lần biến thứ hai là `PAY_1` (trạng thái trả nợ tháng gần nhất, 6,4%). Trên Summary Plot, `PAY_MAX` tách thành hai cụm rõ rệt: giá trị thấp cho SHAP ≈ −0,5, giá trị cao cho SHAP từ +0,3 đến +1,7. Riêng cột `PAY_1_2` (trễ 2 tháng ở tháng gần nhất) có thể làm tăng log-odds tới khoảng +1,4.
- **Hành vi trả nợ và mức sử dụng hạn mức là nhóm quan trọng tiếp theo.** Số tiền trả `PAY_AMT1`, `PAY_AMT2`, `PAY_AMT6` và tỷ lệ trả/dư nợ `PAY_RATIO_4`, `PAY_RATIO_5` có quan hệ nghịch với rủi ro: giá trị cao (màu đỏ) tập trung ở vùng SHAP âm. Ngược lại, tỷ lệ sử dụng hạn mức `UTIL_1`–`UTIL_3` có quan hệ thuận: giá trị cao làm tăng rủi ro. `LIMIT_BAL` cao nhìn chung làm giảm rủi ro.
- **Dư nợ tuyệt đối có quan hệ không đơn điệu.** `BILL_AMT1`, `BILL_AMT3` đứng trong Top 10 nhưng màu phân bố đan xen ở cả hai phía trục 0. Đóng góp của các biến này phụ thuộc vào `LIMIT_BAL` và `UTIL_*` (cùng một mức dư nợ mang ý nghĩa khác nhau tùy hạn mức), đồng thời sáu biến `BILL_AMT1`–`6` tương quan rất cao với nhau. `BILL_DELTA` cao (biến động dư nợ lớn) có xu hướng làm giảm rủi ro — chiều tác động này cần được kiểm tra thêm bằng Dependence Plot.
- **Mức độ quan trọng phân bố rộng.** Top 10 biến gốc chiếm 46% và Top 20 chiếm 70% tổng mean(|SHAP|); phần còn lại trải trên 29 biến. Trên Bar Plot theo cột, 30/119 cột có mean(|SHAP|) = 0, chủ yếu là các mức hiếm của `PAY_x` (trễ ≥ 4 tháng) không được mô hình sử dụng để phân nhánh.
- **Biến nhân khẩu học có đóng góp thấp.** `EDUCATION`, `MARRIAGE`, `AGE_BIN` xếp hạng 32–34/49, tổng cộng 4,4%. Họ đặc trưng cờ trả tối thiểu `MIN_PAY_FLAG_*` chỉ chiếm khoảng 1%.

### 7.3. Đối chiếu với IV và ảnh hưởng của tương quan

Thứ hạng SHAP khác đáng kể với thứ hạng IV ở nhóm biến trễ hạn. `PAY_LATE_2PLUS_COUNT` (IV = 0,86, hạng 2 theo IV), `PAY_MEAN` (IV = 0,76) và `PAY_LATE_CONSECUTIVE` (IV = 0,85) chỉ xếp hạng 31–41 theo SHAP; `PAY_2`–`PAY_6` (IV 0,29–0,55) xếp hạng 37–43. Các biến này mang thông tin gần như trùng lặp với `PAY_MAX` và `PAY_1`; mô hình cây chọn một đại diện để phân nhánh và SHAP dồn phần lớn đóng góp cho đại diện đó. Đây là biểu hiện của hiện tượng đã nêu tại `project_plan.md` mục 3.5e: SHAP phân bổ đóng góp giữa các biến tương quan cao một cách tùy ý. Do đó, mức độ quan trọng thấp của một biến riêng lẻ trong nhóm này không đồng nghĩa với việc biến đó không mang thông tin dự báo. Khi gộp theo họ đặc trưng, các biến lịch sử trễ hạn (`PAY_1`–`6` và các biến tổng hợp trễ hạn) chiếm khoảng 29% tổng mức độ quan trọng, lớn nhất trong các họ.

### 7.4. Độ ổn định

Thứ hạng mức độ quan trọng theo biến gốc gần như không đổi khi thay mẫu: hệ số Spearman ≥ 0,998 với hai mẫu Valid khác và với mẫu từ tập Train, Top 10 trùng 9–10/10 biến. Cỡ mẫu 3.000 dòng là đủ để ước lượng mức độ quan trọng toàn cục, và không có dấu hiệu khác biệt giữa hành vi của mô hình trên dữ liệu huấn luyện và dữ liệu ngoài huấn luyện ở cấp độ toàn cục.

### 7.5. Giới hạn

- Kết quả áp dụng cho XGBoost tham số mặc định trước tuning; thứ hạng có thể thay đổi sau khi tuning bằng Optuna và áp dụng monotonic constraints (Tuần 2 – T4/T5). Phân tích cần được chạy lại trên mô hình cuối.
- Giá trị SHAP ở thang margin của mô hình chưa hiệu chuẩn. Với Platt scaling $\text{PD} = \sigma(a\,m + b)$, giá trị SHAP trên thang logit(PD) bằng $a \cdot \phi_i$, nên thứ hạng mức độ quan trọng không thay đổi (`project_plan.md` mục 3.5e).
- Summary Plot không thể hiện tương tác giữa các đặc trưng; quan hệ không đơn điệu của `BILL_AMT*` và chiều tác động của `BILL_DELTA` cần được làm rõ bằng Dependence Plot.

### 7.6. Đầu vào cho các bước tiếp theo

- **Dependence Plot (Tuần 2 – T4):** ưu tiên `PAY_MAX`, `PAY_1`, `PAY_AMT1`, `PAY_AMT2`, `BILL_AMT1`, `BILL_DELTA`, `PAY_RATIO_5`, `UTIL_1`, `LIMIT_BAL`.
- **Nhóm đặc trưng cho reason codes (Tuần 2 – T4):** kết quả mục 7.3 củng cố quyết định xây dựng reason codes theo nhóm thay vì từng biến; các biến trễ hạn cần được gộp vào cùng một nhóm.
- **Lọc đặc trưng (Tuần 2 – T4, thành viên A):** họ `MIN_PAY_FLAG_*` (≈ 1%) và các biến `PAY_LATE_*`, `PAY_2`–`PAY_6` có đóng góp SHAP thấp do trùng lặp thông tin có thể được xem xét khi loại biến tương quan cao.

## Hình xuất ra

Các hình được lưu tại `reports/figures/` (không commit, sinh lại bằng cách chạy notebook):

- `shap_xgb_summary.png` — Summary Plot (beeswarm), Top 20 cột.
- `shap_xgb_bar.png` — Bar Plot mean(|SHAP|) theo cột sau tiền xử lý.
- `shap_xgb_bar_by_variable.png` — Bar Plot mean(|SHAP|) theo biến gốc.
- `shap_xgb_importance_by_variable.csv` — bảng mean(|SHAP|), share, thứ hạng theo biến gốc.